# NB_hospital_daily_summary
Builds **`<catalog>.gold.hospital_daily_summary`** from the silver layer, per STTM *Silver to Gold (CareSync Health Network) v1.0*.

* **Grain:** one row per hospital per calendar day (days with lab activity)
* **Driving table:** `<catalog>.silver.lab_results` grouped by `hospital_id`, `test_date`
* **Joins (LEFT):** `silver.hospitals` on `hospital_id`; `silver.calendar` on `date_key = date_format(test_date, 'yyyyMMdd')` (cast to INT, since `calendar.date_key` is INT)
* **Window:** `LAG(total_tests)` partitioned by `hospital_id`, ordered by `summary_date` (day-over-day trend)
* **Filters:** none specified
* **Load type:** FULL, `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
import re

dbutils.widgets.text("catalog", "adb_caresync_apollo", "Catalog")
catalog = dbutils.widgets.get("catalog").strip()

# Guard against injection: catalog must be a plain identifier
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError(f"Invalid catalog name: {catalog!r}")

silver = f"`{catalog}`.silver"
gold = f"`{catalog}`.gold"
target_table = f"{gold}.hospital_daily_summary"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")
print(f"Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per hospital per day with lab-test volume, quality rates and day-over-day trend. Source: STTM silver_to_gold v1.0'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                             AS summary_date,
    COUNT(*)                              AS total_tests,
    COUNT(DISTINCT patient_id)            AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)             AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM {silver}.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)             AS hospital_id,
  CAST(h.hospital_name AS STRING)        AS hospital_name,
  CAST(w.summary_date AS DATE)           AS summary_date,
  CAST(c.day_name AS STRING)             AS day_name,
  CAST(c.is_weekend AS BOOLEAN)          AS is_weekend,
  CAST(w.total_tests AS INT)             AS total_tests,
  CAST(w.unique_patients_tested AS INT)  AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)     AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)     AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)     AS critical_test_count,
  CAST(w.abnormal_test_count / w.total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(w.critical_test_count / w.total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)    AS prev_day_total_tests,
  CAST(
    CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
         ELSE (w.total_tests - w.prev_day_total_tests) / w.prev_day_total_tests * 100
    END AS DECIMAL(6,2))                 AS test_volume_change_pct
FROM with_lag w
LEFT JOIN {silver}.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN {silver}.calendar c
  ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)
""")

In [0]:
updated_df = spark.table(target_table)
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))